# 유사 투수 매칭 (hclust k3 적용 버전)

`02_SA_similar_pitcher_matching.ipynb`를 실제 클러스터 데이터로 적용하고 수정한 버전이다.
02는 예시 json(3건)으로 로직을 검증한 기록용으로 남겨둔다.

> **최종 코드는 `src/utils/find_nearest_pitcher.py`에 있다.** 이 노트북은 이름 기준으로 실험한 기록이며,
> 최종 모듈은 동명이인 문제로 **MLBID 기준**으로 바뀌었고 FIP 조건과 rank 입력이 추가되었다. 맨 아래 Step 6 참고.

## 목적
투수 이름과 연도를 입력하면, 아래 조건을 만족하는 다른 투수 중
투구 폼(릴리스 위치, 익스텐션, 팔각도)이 가장 비슷한 투수-시즌을 찾는다.

## 필터링 조건
1. 같은 클러스터 소속 (`hclust_pitcher_clustered_k3.json` 기준)
2. 평균 구속 차이 ±2mph 이내
3. 3. FIP가 더 낮음 — Step 1~5 실험에는 미적용(당시 데이터 없음), 최종 모듈(Step 6)에는 적용.
   FIP는 `04_SA_fip_collection.ipynb`에서 만든 `data/processed/fip_2021_2025.csv`를 사용한다.
4. 같은 선수는 연도와 관계없이 제외, 한 선수당 가장 가까운 연도 하나만 표시

## 유사도 기준
release_pos_x_arm, release_pos_z, release_extension, arm_angle 4개 변수를 표준화한 뒤
유클리드 거리가 가장 짧은 순으로 정렬한다.

## 사용 데이터
- `data/processed/hclust_pitcher_clustered_k3.json`: 투수-시즌별 평균구속, 클러스터 번호 (팀 제공)
- `data/raw/statcast_*.csv`: 투수-시즌별 투구 폼 변수 계산용 원본

## Step 1. Statcast 원본 불러오기

투구 폼 변수는 json에 없으므로 원본에서 직접 계산한다.
속도를 위해 필요한 7개 컬럼(player_name, game_year, p_throws, release_pos_x,
release_pos_z, release_extension, arm_angle)만 읽는다.
구종 구분 없이 그 투수의 전체 투구 평균을 폼 특성으로 사용한다.

In [7]:
import pandas as pd
import numpy as np
import glob

files = glob.glob("../data/raw/statcast_*.csv")
use_cols = ['player_name', 'game_year', 'p_throws', 'release_pos_x', 'release_pos_z',
            'release_extension', 'arm_angle']

df_all = pd.concat([pd.read_csv(f, usecols=use_cols) for f in files], ignore_index=True)
print(df_all.shape)

(3846144, 7)


## Step 2. 클러스터 json 불러오기

`{선수명: {연도: {average_velocity, cluster}}}` 형식이며, 선수 1,350명이 들어 있다.
k6 버전을 쓰려면 `CLUSTER_FILE` 변수만 바꾸면 된다.

In [8]:
import json

CLUSTER_FILE = 'hclust_pitcher_clustered_k3.json'   # k6으로 바꾸려면 여기만 수정

with open(f'../data/processed/{CLUSTER_FILE}', 'r', encoding='utf-8') as f:
    cluster_json = json.load(f)

records = []
for player, years in cluster_json.items():
    for year, info in years.items():
        records.append({
            'player_name_json': player,
            'game_year': int(year),
            'average_velocity': info['average_velocity'],
            'cluster': info['cluster']
        })
df_json = pd.DataFrame(records)
print(df_json.shape)
print(df_json.head())

(3465, 4)
  player_name_json  game_year  average_velocity  cluster
0       A.J. Alexy       2021              93.3        3
1       A.J. Alexy       2022              94.2        3
2      A.J. Minter       2021              89.0        2
3      A.J. Minter       2022              96.6        3
4      A.J. Minter       2023              95.9        3


## Step 3. arm_angle 컬럼 확인

Statcast 변수설명서(2021년 파일 기준)에는 arm_angle이 "값 존재: 0"으로 기록되어 있었다.
연도별로 실제 값이 채워져 있는지 확인이 필요하다.

In [9]:
arm_check = df_all.groupby('game_year')['arm_angle'].apply(lambda x: x.notna().sum())
total_check = df_all.groupby('game_year')['arm_angle'].size()
print(pd.DataFrame({'전체': total_check, 'arm_angle_존재': arm_check}))

               전체  arm_angle_존재
game_year                      
2021       765733        697923
2022       775330        715496
2023       774038        722962
2024       760248        718992
2025       770795        722894


## Step 4. 투수-시즌별 프로필 집계 + json 병합

release_pos_x는 포수 시점 좌표라 우완은 음수, 좌완은 양수로 찍힌다.
그대로 쓰면 폼이 비슷해도 좌완과 우완 사이 거리가 크게 나오므로,
우완의 부호를 반전해 암사이드 기준(release_pos_x_arm)으로 통일한다.

Statcast 이름은 "성, 이름", json은 "이름 성" 형식이라 병합 전에 통일한다.
이름 기준 병합이라 동명이인은 구분되지 않는다.

In [10]:
def convert_name(name):
    if ',' not in name:
        return name
    last, first = [s.strip() for s in name.split(',', 1)]
    return f"{first} {last}"

# release_pos_x를 암사이드 기준으로 통일 (우완은 부호 반전)
df_all['release_pos_x_arm'] = np.where(df_all['p_throws'] == 'L',
                                        df_all['release_pos_x'], -df_all['release_pos_x'])

agg_cols = ['release_pos_x_arm', 'release_pos_z', 'release_extension', 'arm_angle']
df_agg = df_all.groupby(['player_name', 'game_year'])[agg_cols].mean().reset_index()

hand = df_all.groupby(['player_name', 'game_year'])['p_throws'].agg(lambda s: s.mode().iat[0])
df_agg = df_agg.merge(hand.reset_index(), on=['player_name', 'game_year'])
df_agg['player_name_std'] = df_agg['player_name'].apply(convert_name)

df_profile = df_json.merge(df_agg, left_on=['player_name_json', 'game_year'],
                            right_on=['player_name_std', 'game_year'], how='left')

matched = df_profile[agg_cols].notna().all(axis=1).sum()
print(f"json 투수-시즌 {len(df_profile)}건 중 프로필 매칭 {matched}건")

json 투수-시즌 3465건 중 프로필 매칭 3465건


In [11]:
# 오타니 2024년 데이터가 statcast에 있는지 확인
ohtani = df_agg[df_agg['player_name_std'].str.contains('Ohtani', na=False)]
print(ohtani['game_year'].value_counts())
print(ohtani['player_name_std'].unique())

game_year
2021    1
2022    1
2023    1
2025    1
Name: count, dtype: int64
<ArrowStringArray>
['Shohei Ohtani']
Length: 1, dtype: str


## Step 5. 유사 투수 매칭 함수

- 입력: 선수 이름, 연도, 구속 허용 범위(기본 ±2mph), FIP 컬럼명(선택), 반환 인원 수
- 투구 폼 데이터가 없으면 "매칭 불가", 조건에 맞는 후보가 없으면 "후보 없음"을 반환한다.
- FIP 데이터가 병합되면 `fip_col='FIP'`로 호출해 조건을 추가한다.

In [12]:
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import euclidean

profile_features = ['release_pos_x_arm', 'release_pos_z', 'release_extension', 'arm_angle']

# 프로필 결측 행 제외
df_profile_valid = df_profile.dropna(subset=profile_features).copy()

scaler_profile = StandardScaler()
df_profile_valid[[c + '_z' for c in profile_features]] = scaler_profile.fit_transform(
    df_profile_valid[profile_features])

def find_nearest_pitcher(player_name, year, velocity_tol=2.0, fip_col=None, top_n=1):
    target_row = df_profile_valid[(df_profile_valid['player_name_json'] == player_name) &
                                    (df_profile_valid['game_year'] == year)]
    if target_row.empty:
        return f"{player_name} {year}: 릴리스 위치/익스텐션/팔각도 데이터 없음 (매칭 불가)"
    target = target_row.iloc[0]

    candidates = df_profile_valid[
        (df_profile_valid['cluster'] == target['cluster']) &
        (df_profile_valid['average_velocity'].between(
            target['average_velocity'] - velocity_tol,
            target['average_velocity'] + velocity_tol)) &
        (df_profile_valid['player_name_json'] != player_name)
    ].copy()

    if fip_col is not None and fip_col in df_profile_valid.columns:
        candidates = candidates[candidates[fip_col] < target[fip_col]]

    if candidates.empty:
        return f"{player_name} {year}와(과) 매칭되는 후보 없음 (조건에 맞는 투수-시즌 없음)"

    z_cols = [c + '_z' for c in profile_features]
    target_vec = target[z_cols].values.astype(float)
    candidates['distance'] = candidates[z_cols].apply(
        lambda row: euclidean(row.values.astype(float), target_vec), axis=1)

    result = candidates.sort_values('distance').drop_duplicates('player_name_json')
    return result.head(top_n)[['player_name_json', 'game_year', 'cluster',
                                'average_velocity', 'distance']]

print(find_nearest_pitcher("Gerrit Cole", 2023, top_n=3))
print(find_nearest_pitcher("Justin Verlander", 2024, top_n=3))
print(find_nearest_pitcher("Shohei Ohtani", 2023, top_n=3))

     player_name_json  game_year  cluster  average_velocity  distance
539    Carlos Estévez       2025        3              95.9  0.198589
2199    Luis Severino       2024        3              96.2  0.204056
1317    Hunter Harvey       2023        3              98.3  0.258027
       player_name_json  game_year  cluster  average_velocity  distance
3276        Tyler Wells       2025        3              92.8  0.833488
1214  Giovanny Gallegos       2022        3              94.3  0.849889
2745     Ross Stripling       2024        3              91.6  0.875694
     player_name_json  game_year  cluster  average_velocity  distance
2023     Kumar Rocker       2024        3              96.0  0.252859
1618      Joe Jiménez       2022        3              95.7  0.343968
2940      Shane Smith       2025        3              95.6  0.374387


## 결과 요약 (출력 삭제 전 기록)

### 설정
- 클러스터 파일: `hclust_pitcher_clustered_k3.json` (선수 1,350명, 계층적 군집 k=3)
- 유사도 변수: release_pos_x_arm(좌우완 부호 통일), release_pos_z, release_extension, arm_angle
- 필터: 같은 클러스터, 평균구속 ±2mph, 같은 선수 제외 (모든 연도), 선수당 가장 가까운 연도 하나만
- - 아래 테스트 결과는 FIP 조건 없이 뽑은 것 (FIP는 최종 모듈에서 적용)

### 테스트 결과 (상위 3명, 유클리드 거리)
| 기준 | 1위 | 2위 | 3위 |
|---|---|---|---|
| Gerrit Cole 2023 | Carlos Estévez 2025 (0.199) | Luis Severino 2024 (0.204) | Hunter Harvey 2023 (0.258) |
| Justin Verlander 2024 | Tyler Wells 2025 (0.833) | Giovanny Gallegos 2022 (0.850) | Ross Stripling 2024 (0.876) |
| Shohei Ohtani 2023 | Kumar Rocker 2024 (0.253) | Joe Jiménez 2022 (0.344) | Shane Smith 2025 (0.374) |

벌랜더는 릴리스 높이와 팔각도가 높은 독특한 폼이라 가장 가까운 후보도 거리가 크다.

### 수정 이력
- release_pos_x를 암사이드 기준으로 통일 (좌우완 부호 차이로 거리가 과대 계산되던 문제)
- 같은 선수의 다른 연도가 매칭되던 문제 수정
- 한 선수가 여러 연도로 중복 출력되던 문제 수정

### 이후 반영 (src/utils/find_nearest_pitcher.py)
- 이름 대신 MLBID 기준 병합 (동명이인 해결), 클러스터 json 경로 `data/preprocessed/pitcher_clustered.json`
- FIP 조건 적용 (`data/processed/fip_2021_2025.csv`, 04 노트북에서 생성)
- 입력 (MLBID, 연도, rank), 반환 (MLBID, 연도) 튜플 하나, 조건에 맞는 후보가 없으면 None
- 투수-시즌 프로필을 `data/processed/pitcher_profile.csv`로 미리 계산해 백엔드가 매번 원본을 읽지 않도록 함

### 남은 과제
- k3/k6 등 최종 클러스터 파일 확정 (범근님 json 기준)
- 최소 이닝 기준, 선발/불펜 구분 여부 (`min_ip` 인자로 지원)


## Step 6. 최종 모듈 테스트 (MLBID 기준)

`src/utils/find_nearest_pitcher.py`를 불러와 확인한다.
`build_profile()`은 데이터가 바뀔 때만 한 번 실행하면 되고, 결과는 `data/processed/pitcher_profile.csv`에 저장된다.
MLBID 예시: Gerrit Cole = 543037

In [13]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.utils.find_nearest_pitcher import build_profile, find_nearest_pitcher, rank_candidates

profile = build_profile()
print(profile.shape, "| FIP 있음:", profile["FIP"].notna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sua08\\OneDrive\\문서\\Desktop\\econ1003\\ysal\\9th-first-project-baseball-1\\data\\preprocessed\\pitcher_clustered.json'

In [1]:
print(find_nearest_pitcher(543037, 2023))            # 1위
print(find_nearest_pitcher(543037, 2023, rank=2))    # 2위
print(find_nearest_pitcher(543037, 2023, rank=999))  # 범위 초과 -> None

# 후보 전체를 이름·FIP와 함께 확인 (검증용)
rank_candidates(543037, 2023)[["player_id", "player_name", "game_year", "average_velocity", "FIP", "distance"]].head()

NameError: name 'find_nearest_pitcher' is not defined

In [2]:
import sys, os, json, glob
import pandas as pd
sys.path.append(os.path.abspath('..'))
from src.utils.find_nearest_pitcher import build_profile, rank_candidates, find_nearest_pitcher

# Statcast에서 이름 -> MLBID 대응표 만들기
ids = pd.concat([pd.read_csv(f, usecols=["pitcher", "player_name"]) for f in glob.glob("../data/raw/statcast_*.csv")])
ids = ids.drop_duplicates()
ids["name"] = ids["player_name"].str.split(", ").str[::-1].str.join(" ")
name_to_id = dict(zip(ids["name"], ids["pitcher"]))

# 예전 이름 기준 json을 MLBID 기준으로 변환 (임시 테스트용)
with open("../data/processed/hclust_pitcher_clustered_k3.json", encoding="utf-8") as f:
    old = json.load(f)
new = {str(name_to_id[n]): v for n, v in old.items() if n in name_to_id}
os.makedirs("../data/preprocessed", exist_ok=True)
with open("../data/preprocessed/pitcher_clustered_TEST.json", "w", encoding="utf-8") as f:
    json.dump(new, f)
print("변환:", len(new), "/", len(old))

# 임시 파일로 프로필 만들어서 테스트 (저장 안 함)
profile = build_profile(cluster_file="../data/preprocessed/pitcher_clustered_TEST.json", save=False)
print(profile.shape, "| FIP 있음:", profile["FIP"].notna().sum())
print(find_nearest_pitcher(543037, 2023, profile=profile))
print(find_nearest_pitcher(543037, 2023, rank=2, profile=profile))
rank_candidates(543037, 2023, profile=profile)[["player_id", "player_name", "game_year", "average_velocity", "FIP", "distance"]].head()

변환: 1350 / 1350
(3455, 16) | FIP 있음: 3455
(650556, 2022)
(608349, 2021)


,player_id,player_name,game_year,average_velocity,FIP,distance
0,650556,"Abreu, Bryan",2022,97.2,2.117956,0.476690
1,608349,"Knebel, Corey",2021,96.3,2.897307,0.492020
2,670970,"Morejon, Adrian",2024,97.4,2.789516,0.543136
3,640451,"Harvey, Hunter",2025,96.1,1.353886,0.591950
4,685107,"Veneziano, Anthony",2024,94.8,2.941480,0.592148
